# Analysis of the Boolean model of cell cycle by Sizek et al.

In this jupyter notebook, we will analyse different aspects of the cell cycle model published here : 
https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1006402

In [ ]:
import maboss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
import os
from tools import load_trajs, draw_graph_from_pandas, compute_circuits, compute_stg_counts

The model files are available in the Boolean_models/cell_cycle/ folder of the tutorial sample project.

**API Note:** Use `os.path.join()` to create cross-platform file paths.

In [ ]:
# TODO: Define the path to model files and create file paths
# path = "./Boolean_models/cell_cycle/"
# bnd_file = os.path.join(path, "intracellular_model.bnd")
# cfg_file = os.path.join(path, "intracellular_model.cfg")

## Simulation of the wild type model

**API Notes:**
- `maboss.load(bnd_file, cfg_file)` loads a Boolean model from .bnd and .cfg files
- `sim.update_parameters(max_time=X)` sets simulation duration 
- `sim.network.set_output([nodes])` specifies which nodes to track in output
- `sim.run()` executes the simulation and returns results
- `result.plot_node_trajectory()` plots node probability evolution over time

We initially load this model, and simulate it for 48 hours, focusing on Cyclin A, Cyclin B, Cyclin E and Caspase 3.

In [ ]:
# TODO: Load the model using maboss.load()

# TODO: Check node's names using sim.network.nodes

# TODO: Set simulation time

# TODO: Set output nodes to track cyclins and caspase

# TODO: Run simulation and store result

# TODO: Plot node trajectories


We can observe here the classic sequence of cyclins activation : Cyclin E, followed by Cyclin A, and finally Cyclin B. 
But what we also observe, is that we very quickly loose this cyclic behavior : MaBoSS computes probability distribution. Since cells are not synchronized, very quickly what we obtain is just the average probability of each cyclin at any time point during the cell cycle. This informs us about the duration of the phases, but not about the sequential oscillations.

Aside the cyclin trajectory, we also observe the behavior of Caspase 3, which slowly increase during the simulation to reach about 5% after 48h. This means an average cell will have 5% chances of dying in 48 hours. We can simulate a longer time frame to see it's behavior on more than 48 hours.

**API Note:** `sim.copy()` creates a copy of the simulation to modify parameters without affecting the original.

In [ ]:
# TODO: Create a copy of the simulation for long-term analysis
# sim_long = sim.copy()

# TODO: Update parameters for longer simulation (480 hours)
# sim_long.update_parameters(max_time=480)

# TODO: Run long-term simulation
# res_long = sim_long.run()

# TODO: Plot long-term trajectories
# res_long.plot_node_trajectory()

Here we get a better idea about the long term trajectory of the activation of Caspase 3 : after 480h (20 days), a cell has 50% chances of dying.

One way to try to see better the cyclins oscillations is to remove one of the two sources of stochasticity of MaBoSS : the transition time, by switching to discrete time simulation.

**API Note:** `discrete_time=1` switches to discrete time mode, removing stochastic transition timing.

In [ ]:
# TODO: Create discrete time simulation
# sim_discrete = sim.copy()

# TODO: Update parameters for discrete time simulation
# sim_discrete.update_parameters(discrete_time=1, max_time=250)

# TODO: Run discrete simulation
# res_discrete = sim_discrete.run()

# TODO: Plot discrete time trajectories
# res_discrete.plot_node_trajectory()

We can indeed see a bit of a second oscillation, but barely. We still have one large source of stochasticity that we can't get rid of : the choice of the next transition, from the asynchronous update that MaBoSS is using.

## Analysis of the model for PhysiBoSS with phenotypes output

Conditions describing the transition to a next phase of the cell cycle are not so obvious : It does not depend on only one cyclin, and we need to prevent some aberrant phenotypes. 
To solve this, we added three new nodes : 
- G0G1 entry : representing the transition from G2M to G0G1
- S_entry : representing the transition from G0G1 to S phase
- G2M_entry : representing the transition from S to G2M

**API Note:** `sim.copy()` creates a copy to modify output nodes without affecting the original simulation.

In [ ]:
# TODO: Create phenotype simulation copy
# sim_phenotypes = sim.copy()

# TODO: Set output to track phase transition nodes

# TODO: Run phenotype simulation

# TODO: Plot phenotype trajectories


We can observe in this simulation the sequence of transition which is expected : Cells first enter G0G1, then go to S, and finally to G2M. And then a new cycle starts.

**API Note:** `result.plot_trajectory()` shows state transitions over time, not just node probabilities.

In [ ]:
# TODO: Plot state trajectory (not just node trajectory)


Looking at the state trajectories, we can see a even more important detail : We start from a phase were no transition is active (\<nil\>), then activate the G0G1_entry. Then we activate S_entry, and subsequently inactivate G0G1_entry. We then activate G2M_entry, and again immediately inactivate S_entry. 
This is important because it gives us a clear information of the state of the system at any time. If for example we inactivated G0G1_entry before activating S_entry, we would end up up a \<nil\> state, without remembering in which phase we were.

## Studying the sequence of transitions, and the possible cell cycles

One way to study the sequence of cell cycle phases would be to only focus on the transitions affecting these nodes, and completely ignore the other transitions. To do this, we need to look at the complete list of transitions, and filter out the transitions that don't interest us. 
First, we need to create a simulation with the display_traj setting active, and also reduce the number of cores used in the simulation to 1, as the display_traj mode only support single-core simulation. We also increase the simulation time, in order to get more transitions to get better statistics.

**API Notes:**
- `display_traj=1` enables trajectory recording
- `thread_count=1` required for trajectory mode
- `result._path` contains path to trajectory files

In [ ]:
# TODO: Create trajectory simulation copy
# sim_phenotypes_trajs = sim_phenotypes.copy()

# TODO: Update parameters for trajectory recording
# sim_phenotypes_trajs.update_parameters(display_traj=1, thread_count=1, max_time=480)

# TODO: Run trajectory simulation
# res_phenotypes_trajs = sim_phenotypes_trajs.run()

Once the simulation has completed, we need to filter the trajectories by the cell cycle transition nodes, and build a simplified state transition matrix where we only have the states composed of these cell cycle transition nodes.

**API Notes:**
- `load_trajs()` extracts trajectory data from MaBoSS output files
- `compute_stg_counts()` builds state transition matrix
- `pd.DataFrame()` creates a matrix for visualization

In [ ]:
# TODO: Define output phenotype nodes
# outputs_phenotype = ["G0G1_entry", "G2M_entry", "S_entry"]

# TODO: Load and process trajectories
# trajs, all_states = load_trajs(res_phenotypes_trajs._path, outputs_phenotype)

# TODO: Compute state transition counts
# stg_counts, state_ids, ids_state = compute_stg_counts(trajs, all_states)

# TODO: Create DataFrame for transition matrix
# data = pd.DataFrame(
#     data=stg_counts,
#     index=state_ids.keys(), columns=state_ids.keys()
# )

# TODO: Display the transition matrix
# data

We can then plot this matrix as a graph, and visualize the possible transition between all this subset of states.

**API Note:** `draw_graph_from_pandas()` visualizes the state transition graph from the pandas DataFrame.

In [ ]:
# TODO: Draw state transition graph
# draw_graph_from_pandas(data)